# Phase 2 - Part 2: Synthetic wind generation

Generate synthetic wind years on the coarse sea grid (125 m u/v) to train and stress-test farm layouts (used in 3a/3b). Tier 0 = block bootstrap, Tier 1 = per-cell AR(1) on climatology anomalies. Final siting evaluation is on hidden real target (a secret dataset).

In [ ]:
import os, sys, warnings
from pathlib import Path
try:
    _here = Path(__vsc_ipynb_file__).resolve().parent   # VS Code sets this
except NameError:
    _here = Path.cwd().resolve()
_root = next(d for d in [_here, *_here.parents]
             if (d / 'part0_dataset_setup' / 'target_loader.py').exists())
os.chdir(_root)
sys.path[:0] = ['.', 'part0_dataset_setup', 'part1_forecast',
                'part2_siting', 'part3_economics']
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import synthetic_generator as sg
hist = sg.load_coarse_history()
print("history:", hist.U.shape, "| sea cells:", hist.lat.size)

## Generate both tiers

A synthetic year is 1460 six-hourly steps (365 days x 4).

In [ ]:
boot = sg.bootstrap_year(hist, block_days=14, n_steps=1460, seed=0)
model = sg.fit_ar_generator(hist)
ar = sg.sample_ar_year(model, n_steps=1460, seed=0)
print("bootstrap:", boot.U.shape, "| ar:", ar.U.shape)

## Realism comparison

In [ ]:
rows = []
for name, (U, V) in {"real": (hist.U, hist.V), "bootstrap": (boot.U, boot.V),
                     "ar": (ar.U, ar.V)}.items():
    s = sg.summary_stats(U, V); s["set"] = name; rows.append(s)
pd.DataFrame(rows)[["set", "mean_ws", "std_ws", "weibull_k", "weibull_c"]]

## Diagnostic figures

In [ ]:
def domain_ws(U, V):
    return np.sqrt(U ** 2 + V ** 2)

def domain_mean_ws(U, V):
    return domain_ws(U, V).mean(axis=1)

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, 35, 60)
for name, (U, V) in {"real": (hist.U, hist.V), "bootstrap": (boot.U, boot.V),
                     "ar": (ar.U, ar.V)}.items():
    ax.hist(domain_ws(U, V).ravel(), bins=bins, density=True, histtype="step",
            lw=1.8, label=name)
ax.set_xlabel("wind speed (m/s)"); ax.set_ylabel("density")
ax.set_title("Marginal wind-speed distribution"); ax.legend()
plt.show()

In [ ]:
def lag1_acf(x):
    x = np.asarray(x, float); x = x - x.mean()
    return float((x[:-1] * x[1:]).sum() / (x * x).sum())

acf = {name: lag1_acf(domain_mean_ws(U, V))
       for name, (U, V) in {"real": (hist.U, hist.V),
                            "bootstrap": (boot.U, boot.V),
                            "ar": (ar.U, ar.V)}.items()}
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(list(acf.keys()), list(acf.values()), color=["C0", "C1", "C2"])
ax.set_ylabel("lag-1 autocorrelation")
ax.set_title("Temporal persistence (domain-mean speed)")
for i, (k, v) in enumerate(acf.items()):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom")
plt.show()
acf

In [ ]:
site = sg.to_site_series(ar, float(np.median(hist.lat)), float(np.median(hist.lon)))
n2w = 14 * 4
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(site["time"][:n2w], site["ws"][:n2w], lw=1.2)
ax.set_xlabel("time"); ax.set_ylabel("wind speed (m/s)")
ax.set_title("AR synthetic single-site series (first 2 weeks)")
fig.autofmt_xdate()
plt.show()
site.head()

In [ ]:
from pathlib import Path
outdir = Path("../../../build/phase2_dataset/synthetic")
for k in range(3):
    sg.save_synth_year(sg.sample_ar_year(model, seed=100 + k), outdir / f"synth_ar_{k}.nc")
print("saved 3 synthetic AR years to", outdir.resolve())